# Dataset de entrenamiento — cross-encoder de relevancia normativa

Este notebook construye el dataset de pares `(consulta, artículo, label)` usado para entrenar el **cross-encoder**: el modelo que recibe una consulta en lenguaje natural de un abogado junto con un artículo candidato, y produce un score de qué tan relevante es ese artículo para esa consulta. Este cross-encoder actúa como **reranker**: no busca en todo el corpus normativo por sí mismo, sino que reordena por relevancia un conjunto de candidatos que trae un modelo de búsqueda previo (**bi-encoder**), quedándose con los artículos realmente pertinentes.

**Contenido:**
1. Recolección del corpus de sentencias
2. Extracción de texto de los PDFs
3. Filtrado de sentencias válidas
4. Extracción estructurada por sentencia
5. Construcción de pares de entrenamiento
6. Procesamiento del corpus completo
7. Dataset final
8. Próximos pasos

## 1. Recolección del corpus

Se recolecta el listado de sentencias desde la lista curada de jurisprudencia laboral de redal.org, filtrando por tema de interés.

In [1]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q beautifulsoup4 requests pypdf

import requests
from bs4 import BeautifulSoup
from io import BytesIO
import pandas as pd

URL = "https://colombia.redal.org/sentencias-materia-laboral/"
TEMAS_OBJETIVO = ["Laboral individual", "Estabilidad laboral"]  # ajustable

resp = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.content, "html.parser")

rows = []
for tr in soup.select("table tr"):
    celdas = tr.find_all("td")
    if len(celdas) < 6:
        continue
    tema = celdas[3].get_text(strip=True)
    if tema not in TEMAS_OBJETIVO:
        continue
    link_tag = celdas[5].find("a")
    if not link_tag:
        continue
    rows.append({
        "corporacion": celdas[0].get_text(strip=True),
        "numero": celdas[2].get_text(strip=True),
        "tema": tema,
        "subtema": celdas[4].get_text(strip=True),
        "url_pdf": link_tag["href"],
    })

df_sentencias = pd.DataFrame(rows).drop_duplicates(subset="url_pdf").reset_index(drop=True)
print(f"Sentencias encontradas: {len(df_sentencias)}")
df_sentencias

Sentencias encontradas: 21


,corporacion,numero,tema,subtema,url_pdf
0,"Corte Suprema de Justicia, Sala Casación Laboral",SL3630/2022,Laboral individual,Pagos constitutivos de salario y pagos no sala...,https://colombia.redal.org/wp-content/uploads/...
1,"Corte Suprema de Justicia, Sala Casación Laboral",SL2858/2022,Laboral individual,Contrato realidad con el Estado,https://colombia.redal.org/wp-content/uploads/...
2,Corte Constitucional,SU-087/2022,Estabilidad laboral,Persona en situación de debilidad manifiesta n...,https://colombia.redal.org/wp-content/uploads/...
3,Corte Constitucional,SU-380/2021,Estabilidad laboral,Derecho a la estabilidad laboral de persona en...,https://colombia.redal.org/wp-content/uploads/...
4,Corte Constitucional,C-038/2021,Laboral individual,Discriminación laboral y estereotipos de géner...,https://colombia.redal.org/wp-content/uploads/...
5,Corte Constitucional,C-103/2021,Laboral individual,Jornada de trabajo en el teletrabajo,https://colombia.redal.org/wp-content/uploads/...
6,Corte Constitucional,T-109/2021,Laboral individual,Derechos laborales de modelo webcam,https://colombia.redal.org/wp-content/uploads/...
7,Corte Constitucional,T-055/2020,Laboral individual,Estabilidad laboral de los prepensionados,https://colombia.redal.org/wp-content/uploads/...
8,Corte Constitucional,C-200/2019,Estabilidad laboral,Estabilidad ocupacional reforzada,https://colombia.redal.org/wp-content/uploads/...
9,Corte Constitucional,T-460/2018,Laboral individual,Licencia por calamidad doméstica,https://colombia.redal.org/wp-content/uploads/...


## 2. Extracción de texto de los PDFs

Se descarga y extrae el texto de cada sentencia con `pdfplumber`, que preserva el espaciado real de las palabras (a diferencia de `pypdf`, que en estos documentos pega las palabras entre sí).

In [2]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q pdfplumber

import pdfplumber

def descargar_texto_pdf(url):
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
        resp.raise_for_status()
        with pdfplumber.open(BytesIO(resp.content)) as pdf:
            texto = " ".join(page.extract_text() or "" for page in pdf.pages)
        return " ".join(texto.split())
    except Exception as e:
        print(f"Error con {url}: {e}")
        return None

df_sentencias["texto"] = df_sentencias["url_pdf"].apply(descargar_texto_pdf)
df_sentencias["texto_ok"] = df_sentencias["texto"].notna()
print(df_sentencias[["numero", "tema", "texto_ok"]])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


         numero                 tema  texto_ok
0   SL3630/2022   Laboral individual      True
1   SL2858/2022   Laboral individual      True
2   SU-087/2022  Estabilidad laboral      True
3   SU-380/2021  Estabilidad laboral      True
4    C-038/2021   Laboral individual      True
5    C-103/2021   Laboral individual      True
6    T-109/2021   Laboral individual      True
7    T-055/2020   Laboral individual      True
8    C-200/2019  Estabilidad laboral      True
9    T-460/2018   Laboral individual      True
10  SU-075/2018   Laboral individual      True
11   T-528/2017   Laboral individual      True
12  SU-049/2017  Estabilidad laboral      True
13   C-005/2017  Estabilidad laboral      True
14   C-636/2016   Laboral individual      True
15   T-864/2014   Laboral individual      True
16   C-593/2014   Laboral individual      True
17  SU-071/2013  Estabilidad laboral      True
18   T-936/2009   Laboral individual      True
19   C-930/2009   Laboral individual      True
20  C-1037/20

## 3. Filtrado de sentencias válidas

No todas las sentencias del listado sirven para este dataset. Las que empiezan con **`C-`** son de control de constitucionalidad — revisan la norma en abstracto y no contienen los hechos de un caso real de un trabajador, así que no aportan una consulta útil. Se conservan solo las de **Casación (`SL`)** y **Tutela/Unificación (`T`/`SU`)**, que sí resuelven un caso concreto. Este filtro se aplica antes de la extracción estructurada, para no gastar llamadas a la API en sentencias que de todas formas se van a descartar.

In [3]:
candidatas = df_sentencias[
    (~df_sentencias["numero"].str.startswith("C-")) &
    (df_sentencias["texto_ok"])
].reset_index(drop=True)

print(f"Sentencias válidas (SL/T/SU, con texto): {len(candidatas)} de {len(df_sentencias)}")
candidatas[["numero", "tema", "subtema"]]

Sentencias válidas (SL/T/SU, con texto): 13 de 21


,numero,tema,subtema
0,SL3630/2022,Laboral individual,Pagos constitutivos de salario y pagos no sala...
1,SL2858/2022,Laboral individual,Contrato realidad con el Estado
2,SU-087/2022,Estabilidad laboral,Persona en situación de debilidad manifiesta n...
3,SU-380/2021,Estabilidad laboral,Derecho a la estabilidad laboral de persona en...
4,T-109/2021,Laboral individual,Derechos laborales de modelo webcam
5,T-055/2020,Laboral individual,Estabilidad laboral de los prepensionados
6,T-460/2018,Laboral individual,Licencia por calamidad doméstica
7,SU-075/2018,Laboral individual,Reglas jurisprudenciales para la aplicación de...
8,T-528/2017,Laboral individual,Traslado de docentes (límites al ius variandi)
9,SU-049/2017,Estabilidad laboral,Derecho a la estabilidad ocupacional reforzada


## 4. Extracción estructurada por sentencia

Cada sentencia se procesa con un modelo de lenguaje (Claude, vía Anthropic API) para extraer, en formato JSON:

- `hechos_resumidos`: los hechos del caso en lenguaje coloquial — esta es la **consulta** de entrenamiento, tal como la formularía un trabajador.
- `pretension`: qué pedía el demandante.
- `articulos_fundamento_directo`: los artículos cuya interpretación fue **determinante** para resolver el punto concreto en disputa. Estos son los únicos que se usan como **positivos** en el dataset.
- `articulos_marco_general`: artículos que la sentencia también cita, pero que son principios generales, reglas de remisión o contexto normativo (ej. favorabilidad, analogía) — no decidieron el punto específico del caso, y casi cualquier sentencia laboral podría citarlos. Se separan de `articulos_fundamento_directo` porque, si se mezclaran, el cross-encoder aprendería a asociar cualquier consulta laboral con artículos genéricos en vez de con los realmente relevantes.
- `decision` y `elemento_no_acreditado`: resultado del caso y, si aplica, qué no logró probarse.

In [4]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q anthropic python-dotenv

import anthropic
import json
import time
import os
from dotenv import load_dotenv

load_dotenv()  # lee ANTHROPIC_API_KEY del .env en la raíz del repo

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

PROMPT_TEMPLATE = """Lee la siguiente sentencia laboral colombiana. Extrae exactamente esta información y devuelve SOLO un JSON válido, sin texto adicional ni backticks:

{{
  "hechos_resumidos": "los hechos del caso en 2-3 líneas, en lenguaje coloquial, como si un trabajador lo contara (ej. 'me despidieron después de...' o 'trabajé X años y...')",
  "pretension": "qué pedía el demandante, en pocas palabras",
  "articulos_fundamento_directo": ["artículos cuya interpretación/aplicación fue DETERMINANTE para resolver el punto concreto en disputa de este caso. Pregunta guía: si se quitara este artículo, ¿cambiaría el razonamiento de por qué se concedió o negó la pretensión específica? Si sí, va aquí. Formato 'CST Art. X' o 'Ley X de YYYY, Art. Y'"],
  "articulos_marco_general": ["artículos que la sentencia cita pero que son principios generales, reglas de interpretación/remisión, o contexto normativo (ej. favorabilidad, analogía, primacía de la realidad, normas constitucionales genéricas) — NO decidieron el punto específico del caso, cualquier sentencia laboral podría citarlos. Mismo formato."],
  "decision": "concedida" o "negada" o "parcial",
  "elemento_no_acreditado": "si la pretensión fue negada o parcial, qué elemento/requisito no se acreditó según el juez; si fue concedida totalmente, deja este campo vacío"
}}

Sentencia:
{texto}
"""

def extraer_estructura(texto, numero, max_chars=15000):
    texto_truncado = texto[:max_chars]  # las sentencias largas se truncan para no exceder contexto/costo
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(texto=texto_truncado)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error extrayendo {numero}: {e}")
        return None

## 5. Construcción de pares de entrenamiento

Por cada sentencia se generan tres tipos de par:

- **Positivo** — uno por cada artículo en `articulos_fundamento_directo`, con `label = 1`.
- **Negativo fácil** — un artículo de un tema laboral claramente distinto (maternidad, vacaciones, jornada, seguridad social), tomado de un pool curado a mano. Es "fácil" porque no tiene relación temática alguna con la consulta, así que no requiere procesar ninguna otra sentencia.
- **Negativo difícil (placeholder)** — un artículo real y temáticamente cercano pero incorrecto, generado con una segunda llamada al LLM. Esto es un placeholder temporal: el negativo difícil real debe salir de **hard-negative mining** con el bi-encoder — correr la consulta contra el corpus normativo completo, tomar los candidatos más similares semánticamente que el bi-encoder trae, y usar los que no sean el artículo correcto. Esa señal solo existe una vez que el bi-encoder esté entrenado, así que por ahora se aproxima con el LLM.

In [5]:
POOL_NEGATIVOS_FACILES = [
    {"articulo": "CST Art. 236", "tema": "licencia de maternidad"},
    {"articulo": "CST Art. 186", "tema": "vacaciones anuales"},
    {"articulo": "CST Art. 161", "tema": "jornada de trabajo"},
    {"articulo": "Ley 100 de 1993, Art. 13", "tema": "seguridad social"},
]

PROMPT_NEGATIVO_DIFICIL = """Dada esta consulta de un caso laboral colombiano:

{consulta}

Y sabiendo que los artículos correctamente aplicables son: {articulos_correctos}

Dame UN artículo real del derecho laboral colombiano (CST, leyes laborales) que esté relacionado temáticamente con la consulta pero que NO sea el fundamento correcto de la decisión — es decir, un artículo que alguien podría confundir con el correcto pero que no aplica aquí.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"articulo_incorrecto": "...", "por_que_se_confunde": "..."}}
"""

def generar_negativo_dificil(consulta, articulos_correctos):
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": PROMPT_NEGATIVO_DIFICIL.format(
                consulta=consulta, articulos_correctos=articulos_correctos)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error generando negativo difícil: {e}")
        return None

def construir_pares(extraccion, numero):
    consulta = extraccion["hechos_resumidos"]
    articulos_correctos = extraccion.get("articulos_fundamento_directo", [])
    articulos_marco = extraccion.get("articulos_marco_general", [])
    ya_citados = set(articulos_correctos) | set(articulos_marco)

    if not articulos_correctos:
        print(f"{numero}: sin articulos_fundamento_directo, se omite")
        return None

    pares = [
        {"consulta": consulta, "articulo": art, "tipo": "positivo", "label": 1}
        for art in articulos_correctos
    ]

    candidatos_faciles = [n for n in POOL_NEGATIVOS_FACILES if n["articulo"] not in ya_citados]
    if candidatos_faciles:
        pares.append({
            "consulta": consulta,
            "articulo": candidatos_faciles[0]["articulo"],
            "tipo": "negativo_facil",
            "label": 0,
        })

    negativo_dificil = generar_negativo_dificil(consulta, articulos_correctos)
    if negativo_dificil:
        pares.append({
            "consulta": consulta,
            "articulo": negativo_dificil["articulo_incorrecto"],
            "tipo": "negativo_dificil_placeholder",
            "label": 0,
        })

    df = pd.DataFrame(pares)
    df["sentencia_origen"] = numero
    return df

## 6. Procesamiento del corpus completo

Se corre la extracción y construcción de pares sobre todas las sentencias válidas. Cada sentencia implica dos llamadas a la API (extracción + negativo difícil), con una pausa entre llamadas para no saturar el rate limit. Las sentencias que fallan en la extracción o que no producen `articulos_fundamento_directo` se omiten sin detener el resto del proceso.

In [6]:
resultados = []
extracciones_fallidas = []

for i, row in candidatas.iterrows():
    print(f"[{i+1}/{len(candidatas)}] Procesando {row['numero']}...")
    extraccion = extraer_estructura(row["texto"], row["numero"])
    time.sleep(1)

    if extraccion is None:
        extracciones_fallidas.append(row["numero"])
        continue

    df_par = construir_pares(extraccion, row["numero"])
    time.sleep(1)

    if df_par is not None:
        resultados.append(df_par)
    else:
        extracciones_fallidas.append(row["numero"])

print(f"\nProcesadas con éxito: {len(resultados)} de {len(candidatas)}")
if extracciones_fallidas:
    print("Omitidas:", extracciones_fallidas)

[1/13] Procesando SL3630/2022...


[2/13] Procesando SL2858/2022...


[3/13] Procesando SU-087/2022...


[4/13] Procesando SU-380/2021...


[5/13] Procesando T-109/2021...


...

[10/13] Procesando SU-049/2017...


[11/13] Procesando T-864/2014...


[12/13] Procesando SU-071/2013...


[13/13] Procesando T-936/2009...



Procesadas con éxito: 10 de 13
Omitidas: ['T-109/2021', 'T-460/2018', 'T-528/2017']


## 7. Dataset final

In [7]:
df_dataset = pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()

print(f"Total de pares: {len(df_dataset)}")
print(f"Sentencias representadas: {df_dataset['sentencia_origen'].nunique()}")
print(df_dataset["tipo"].value_counts())
df_dataset.head()

Total de pares: 43
Sentencias representadas: 10
tipo
positivo                        23
negativo_facil                  10
negativo_dificil_placeholder    10
Name: count, dtype: int64


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL3630/2022
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL3630/2022
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL3630/2022
3,Trabajé como operador de bus articulado desde ...,CST Art. 130,negativo_dificil_placeholder,0,SL3630/2022
4,Trabajé para el ISS desde septiembre de 2000 h...,"Decreto 2127 de 1945, Art. 20",positivo,1,SL2858/2022


In [ ]:
import re

def normalizar_sentencia(s):
    # unifica el formato de año a 2 dígitos (ej. "SU-087/2022" -> "SU-087/22"),
    # igual al que usan los notebooks ds_parte2_corte_const y ds_parte3_research_list
    m = re.match(r"([A-Z]+)-?(\d+)/(\d+)$", str(s))
    if not m:
        return s
    tipo, numero, anio = m.groups()
    return f"{tipo}-{numero}/{anio[-2:]}"

df_dataset["sentencia_origen"] = df_dataset["sentencia_origen"].apply(normalizar_sentencia)

OUTPUT_PATH = "../../data/dataset_cross_encoder.csv"

# dataset_cross_encoder.csv es el archivo CONSOLIDADO (este notebook + ds_parte2_corte_const.ipynb +
# ds_parte3_research_list.ipynb). Si ya existe, solo se reemplazan las filas que salieron de ESTE
# notebook (por sentencia_origen) — así una re-corrida nunca borra lo que aportaron los otros dos.
if os.path.exists(OUTPUT_PATH):
    df_existente = pd.read_csv(OUTPUT_PATH)
    sentencias_de_este_run = set(df_dataset["sentencia_origen"])
    df_otros = df_existente[~df_existente["sentencia_origen"].isin(sentencias_de_este_run)]
    df_final = pd.concat([df_otros, df_dataset], ignore_index=True)
else:
    df_final = df_dataset

df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado en {OUTPUT_PATH} ({len(df_final)} filas totales)")

## 8. Próximos pasos

- **Hard-negative mining real:** reemplazar `generar_negativo_dificil` por negativos extraídos de la salida del bi-encoder sobre el corpus normativo completo, una vez esté entrenado.
- **Ampliar el corpus:** las sentencias de redal.org son un conjunto curado pero pequeño. Fuentes adicionales a explorar: el sistema de consulta de jurisprudencia oficial de la Sala de Casación Laboral, o ampliar las categorías de tema dentro del propio redal.org.
- **Balance de clases:** revisar la proporción positivo/negativo del `df_dataset` final y ajustar el pool de negativos fáciles o el muestreo si hace falta más variedad.